# RSO-112: Analyse thermal and stability indicators during Dome louver testing

During the shutdown period (31.1. -14.2) we run multiple versions (different louver configurations) of the BLOCK-T679. Even though we were not on sky and we don’t have the IQ data, we would like to see the inside dome temperature response for different louvers configurations. 

Analyze dome and telescope thermal response during the 6-louver experimental campaigns, focusing on temperature coupling and ventilation behavior in the absence of image-quality (FWHM) indicators.

**Descripcion**

Create a small set of easy-to-interpret temperature indicators that describe how the dome and telescope behave thermally during each louvers configuration.

 

**Expected results:**

New variables:

inside vs outside temperature difference

telescope vs outside temperature difference

temperature gradient across the telescope sensors (111-113)

Table: basic statistics per configuration (average, variation)

Plots: simple box or line plots showing how these indicators change between configurations

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from astropy.time import Time


from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState

In [ ]:
t_start_period = Time("2026-01-31T00:00:00Z", scale="utc")
t_end_period = Time("2026-02-15T00:00:00Z", scale="utc")

efd_client = makeEfdClient()

In [ ]:
# make a list of all topics in the EFD related to MTMount
topics = await efd_client.get_topics()
for topic in topics:
    if 'MTDome' in topic:
        print(topic)

# Queries

In [ ]:
def query_setlouvers(start, end):
    df_louvers = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.command_setLouvers",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_louvers

# Configuration of louvers

In [ ]:
df_setlouvers = query_setlouvers(t_start_period, t_end_period)

In [ ]:
df_setlouvers.head()

In [ ]:
# Copy current index into a new column before any merge
df_setlouvers['time_stamp'] = df_setlouvers.index

In [ ]:
# Select all columns that start with "position"
position_cols = df_setlouvers.filter(regex=r'^position').columns

# Sort columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position', '')))

# Compute unique combinations
combination_counts = (
    df_setlouvers[position_cols]
    .value_counts()
    .reset_index(name='count')
)

# Create configuration ID (1 to N)
combination_counts['louvers_conf'] = range(1, len(combination_counts) + 1)

# Merge configuration ID back into original dataframe
df_setlouvers = df_setlouvers.merge(
    combination_counts[position_cols + ['louvers_conf']],
    on=position_cols,
    how='left'
)

print(f"Number of unique configurations detected: {len(combination_counts)}\n")

# Print configurations showing only non-zero positions
for _, row in combination_counts.iterrows():
    
    conf_id = row['louvers_conf']
    count = row['count']
    
    print(f"Configuration {conf_id} (appears {count} times):")
    
    # Extract position values
    config = row[position_cols]
    
    # Keep only non-zero values
    non_zero = config[config != 0]
    
    if len(non_zero) == 0:
        print("  All positions are 0")
    else:
        for col, val in non_zero.items():
            print(f"  {col}: {val}")
    
    print("-" * 40)

15 combinations have been made, varying the opening of the louvers: 2, 11, 12, 20, 21, and 29.

In [ ]:
df_setlouvers.head()

# **********

In [ ]:
def query_thermal(start, end):
    df_thermal = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.thermal",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_thermal

In [ ]:
df_thermal = query_thermal(t_start_period, t_end_period)

In [ ]:
df_thermal.head()

In [ ]:
df.columns

In [ ]:
# Make sure time columns are datetime (if not already)
df_setlouvers['time_stamp'] = pd.to_datetime(df_setlouvers['time_stamp'])
df_thermal['timestamp'] = pd.to_datetime(df_thermal['timestamp'])

# Sort both dataframes by time
df_setlouvers = df_setlouvers.sort_values('time_stamp')
df_thermal = df_thermal.sort_values('timestamp')

In [ ]:
# Perform temporal merge
df_thermal_with_conf = pd.merge_asof(
    df_thermal,
    df_setlouvers[['time_stamp', 'louvers_conf']],
    left_on='timestamp',
    right_on='time_stamp',
    direction='backward'  # take last known configuration
)

In [ ]:
def query_weather(start, end):
    df_weather = getEfdData(
        client=efd_client,
        topic="lsst.sal.WeatherForecast.dailyTrend",
        columns=["Max temperature", "temperatureMin", "Mean temperature", ],
        begin=start,
        end=end,
    )

    return df_weather

In [ ]:
df = query_weather(t_start_period, t_end_period)

## *****************************

In [ ]:
# Esto debe tener demasiado datos y no carga ni cambiando las fechas
def logevent_louversEnabled(start, end):
    df_louvers = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.louvers",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_louvers

In [ ]:
df = logevent_louversEnabled(t_start_period, t_end_period)

In [ ]:
df

In [ ]:
## Insert here the dayObs of interest
dayObs = 20260208# 20231122

In [ ]:
# Select data from a given date
eventMaker = TMAEventMaker()
events = eventMaker.getEvents(dayObs)


# Get events related to soak tests (block 137 currently)
blockT679 = []
for event in events:
    blockInfos = event.blockInfos
    if blockInfos is None:
        continue  # no block info attached to event at all

    # check if any of the attached blockInfos are for block 137
    blockNums = {b.blockNumber for b in blockInfos}
    print(blockNums)
    if T679 in blockNums:
        blockT679.append(event)

print(f"Of the {len(events)} events, {len(blockT679)} relate to block BLOCK-T679.")

In [ ]:
from lsst.summit.utils.blockUtils import BlockParser
from lsst.summit.utils.tmaUtils import TMAEventMaker

In [ ]:
# Set the day_obs list 
day_obs_list = range(20250127, 20260215 + 1)

# For the TMA events 
event_maker = TMAEventMaker()

In [ ]:
# For each day_obs in the list determine which blocks were run and put
# the list of blocks into the block_list.
 
block_list = []

for day_obs in day_obs_list:
    block_parser = BlockParser(day_obs)
    blocks = block_parser.getBlockNums()
    block_list.append(blocks)

# Put the variable length nested list into an awkward array and then 
# put that into a pandas dataframe with the awkward array extension
# so that the list of blocks is shown in a column.
blocks = ak.Array({"day_obs": day_obs_list, "blocks": block_list})
series = akpd.from_awkward(blocks)
pandas_df = series.ak.to_columns(extract_all=True)
pandas_df

In [ ]:
def day_obs_report(day_obs):
    '''
    Loop over the blocks and sequences for one day and produce a report.
    Interspace TMA events with the block info.
    '''

    block_parser = BlockParser(day_obs)
    tma_events = event_maker.getEvents(day_obs)
    blocks = block_parser.getBlockNums()

    print(f'SUMMARY REPORT FOR DAYOBS: {day_obs} \n')
    print(blocks)
    for block_id in blocks:
        sequences =  block_parser.getSeqNums(block_id)

        print(f'BLOCK:SEQ \t STATES')

        for seq_id in sequences:
            info = block_parser.getBlockInfo(block_id, seq_id)
            state_string = ' '.join([str(state) for state in info.states])
            print(f'{info.blockNumber}:{info.seqNum} \t\t {state_string}')

            # Also print any TMA events for this block/sequence
            event = block_parser.getEventsForBlock(tma_events, block_id, seq_id)
            if event: print(event)

        print(f'\n')

In [ ]:
day_obs_list = range(20260201, 20260214 + 1)

for day_obs in day_obs_list:
    day_obs_report(day_obs)